In [25]:
import duckdb
import pandas as pd

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 160)

con = duckdb.connect(r'data\gold.duckdb', read_only=True)
print('Connected to gold.duckdb')

def q(sql):
    return con.sql(sql).df()

Connected to gold.duckdb


In [26]:
# ── Change player name here to analyse anyone ────────────────────────────────
PLAYER = "MA Starc"
# ────────────────────────────────────────────────────────────────────────────
print(f"Showing IPL stats for: {PLAYER}")

Showing IPL stats for: MA Starc


## Batting

In [27]:
# Career IPL batting summary
q(f"""
WITH bat AS (
    SELECT d.match_id, d.innings_number,
        sum(d.runs_batter)                                                              AS runs,
        sum(CASE WHEN NOT d.extras_wides THEN 1 ELSE 0 END)                            AS balls,
        sum(CASE WHEN d.runs_batter = 4 AND NOT d.runs_non_boundary THEN 1 ELSE 0 END) AS fours,
        sum(CASE WHEN d.runs_batter = 6 AND NOT d.runs_non_boundary THEN 1 ELSE 0 END) AS sixes
    FROM main_marts.fct_deliveries d
    JOIN main_marts.fct_match_results m USING (match_id)
    WHERE d.batter = '{PLAYER}'
      AND m.event_name ILIKE '%indian premier league%'
      AND d.innings_number IN (1, 2)
    GROUP BY d.match_id, d.innings_number
),
dis AS (
    SELECT DISTINCT d.match_id, d.innings_number
    FROM main_marts.fct_deliveries d
    JOIN main_marts.fct_match_results m USING (match_id)
    WHERE d.is_wicket AND d.wicket_player_out = '{PLAYER}'
      AND m.event_name ILIKE '%indian premier league%'
      AND d.innings_number IN (1, 2)
),
all_inn AS (
    SELECT match_id, innings_number FROM bat
    UNION
    SELECT match_id, innings_number FROM dis
    UNION
    -- Player reached the crease as non-striker but innings ended before they faced a ball.
    -- Only included when the player does not bat anywhere else in the same match,
    -- which prevents Cricsheet data errors (wrong non_striker in opposing team's innings).
    SELECT DISTINCT d.match_id, d.innings_number
    FROM main_marts.fct_deliveries d
    JOIN main_marts.fct_match_results m USING (match_id)
    WHERE d.non_striker = '{PLAYER}'
      AND m.event_name ILIKE '%indian premier league%'
      AND d.innings_number IN (1, 2)
      AND NOT EXISTS (
          SELECT 1 FROM main_marts.fct_deliveries d2
          WHERE d2.match_id = d.match_id AND d2.batter = '{PLAYER}'
      )
),
inn AS (
    SELECT
        a.match_id, a.innings_number,
        coalesce(b.runs,  0) AS runs,
        coalesce(b.balls, 0) AS balls,
        coalesce(b.fours, 0) AS fours,
        coalesce(b.sixes, 0) AS sixes,
        (d.match_id IS NOT NULL)::integer AS dismissed
    FROM all_inn a
    LEFT JOIN bat b USING (match_id, innings_number)
    LEFT JOIN dis d USING (match_id, innings_number)
)
SELECT
    count(DISTINCT match_id)                                    AS mat,
    count(*)                                                    AS inns,
    sum(1 - dismissed)                                         AS no,
    sum(runs)                                                   AS runs,
    max(runs)                                                   AS hs,
    round(sum(runs)::double / nullif(sum(dismissed), 0), 2)    AS ave,
    sum(balls)                                                  AS bf,
    round(sum(runs) * 100.0 / nullif(sum(balls), 0), 2)        AS sr,
    sum(CASE WHEN runs >= 100 THEN 1 ELSE 0 END)               AS "100",
    sum(CASE WHEN runs >= 50 AND runs < 100 THEN 1 ELSE 0 END) AS "50",
    sum(CASE WHEN runs = 0 AND dismissed = 1 THEN 1 ELSE 0 END) AS "0",
    sum(fours)                                                  AS "4s",
    sum(sixes)                                                  AS "6s"
FROM inn
""")

,mat,inns,no,runs,hs,ave,bf,sr,100,50,0,4s,6s
0,23,23,12.0,111.0,29.0,10.09,119.0,93.28,0.0,0.0,3.0,11.0,0.0


In [28]:
# IPL batting by season
q(f"""
WITH bat AS (
    SELECT d.match_id, d.innings_number, m.season, d.batting_team,
        sum(d.runs_batter)                                                              AS runs,
        sum(CASE WHEN NOT d.extras_wides THEN 1 ELSE 0 END)                            AS balls,
        sum(CASE WHEN d.runs_batter = 4 AND NOT d.runs_non_boundary THEN 1 ELSE 0 END) AS fours,
        sum(CASE WHEN d.runs_batter = 6 AND NOT d.runs_non_boundary THEN 1 ELSE 0 END) AS sixes
    FROM main_marts.fct_deliveries d
    JOIN main_marts.fct_match_results m USING (match_id)
    WHERE d.batter = '{PLAYER}'
      AND m.event_name ILIKE '%indian premier league%'
      AND d.innings_number IN (1, 2)
    GROUP BY d.match_id, d.innings_number, m.season, d.batting_team
),
dis AS (
    SELECT DISTINCT d.match_id, d.innings_number, m.season, d.batting_team
    FROM main_marts.fct_deliveries d
    JOIN main_marts.fct_match_results m USING (match_id)
    WHERE d.is_wicket AND d.wicket_player_out = '{PLAYER}'
      AND m.event_name ILIKE '%indian premier league%'
      AND d.innings_number IN (1, 2)
),
all_inn AS (
    SELECT match_id, innings_number, season, batting_team FROM bat
    UNION
    SELECT match_id, innings_number, season, batting_team FROM dis
    UNION
    SELECT DISTINCT d.match_id, d.innings_number, m.season, d.batting_team
    FROM main_marts.fct_deliveries d
    JOIN main_marts.fct_match_results m USING (match_id)
    WHERE d.non_striker = '{PLAYER}'
      AND m.event_name ILIKE '%indian premier league%'
      AND d.innings_number IN (1, 2)
      AND NOT EXISTS (
          SELECT 1 FROM main_marts.fct_deliveries d2
          WHERE d2.match_id = d.match_id AND d2.batter = '{PLAYER}'
      )
),
inn AS (
    SELECT
        a.match_id, a.innings_number, a.season, a.batting_team,
        coalesce(b.runs,  0) AS runs,
        coalesce(b.balls, 0) AS balls,
        coalesce(b.fours, 0) AS fours,
        coalesce(b.sixes, 0) AS sixes,
        (d.match_id IS NOT NULL)::integer AS dismissed
    FROM all_inn a
    LEFT JOIN bat b USING (match_id, innings_number)
    LEFT JOIN dis d USING (match_id, innings_number)
)
SELECT
    season,
    mode(batting_team)                                          AS team,
    count(DISTINCT match_id)                                    AS mat,
    count(*)                                                    AS inns,
    sum(1 - dismissed)                                         AS no,
    sum(runs)                                                   AS runs,
    max(runs)                                                   AS hs,
    round(sum(runs)::double / nullif(sum(dismissed), 0), 2)    AS ave,
    sum(balls)                                                  AS bf,
    round(sum(runs) * 100.0 / nullif(sum(balls), 0), 2)        AS sr,
    sum(CASE WHEN runs >= 100 THEN 1 ELSE 0 END)               AS "100",
    sum(CASE WHEN runs >= 50 AND runs < 100 THEN 1 ELSE 0 END) AS "50",
    sum(CASE WHEN runs = 0 AND dismissed = 1 THEN 1 ELSE 0 END) AS "0",
    sum(fours)                                                  AS "4s",
    sum(sixes)                                                  AS "6s"
FROM inn
GROUP BY season
ORDER BY season
""")

,season,team,mat,inns,no,runs,hs,ave,bf,sr,100,50,0,4s,6s
0,2014,Royal Challengers Bangalore,9,9,3.0,85.0,29.0,14.17,84.0,101.19,0.0,0.0,0.0,9.0,0.0
1,2015,Royal Challengers Bangalore,3,3,2.0,11.0,9.0,11.00,14.0,78.57,0.0,0.0,0.0,1.0,0.0
2,2024,Kolkata Knight Riders,5,5,3.0,9.0,6.0,4.50,11.0,81.82,0.0,0.0,2.0,1.0,0.0
3,2025,Delhi Capitals,6,6,4.0,6.0,2.0,3.00,10.0,60.00,0.0,0.0,1.0,0.0,0.0


In [29]:
# Top 20 highest IPL innings
q(f"""
WITH bat AS (
    SELECT d.match_id, d.innings_number, d.match_date, d.batting_team, d.bowling_team,
        m.season,
        sum(d.runs_batter)                                                              AS runs,
        sum(CASE WHEN NOT d.extras_wides THEN 1 ELSE 0 END)                            AS balls,
        sum(CASE WHEN d.runs_batter = 4 AND NOT d.runs_non_boundary THEN 1 ELSE 0 END) AS fours,
        sum(CASE WHEN d.runs_batter = 6 AND NOT d.runs_non_boundary THEN 1 ELSE 0 END) AS sixes
    FROM main_marts.fct_deliveries d
    JOIN main_marts.fct_match_results m USING (match_id)
    WHERE d.batter = '{PLAYER}'
      AND m.event_name ILIKE '%indian premier league%'
      AND d.innings_number IN (1, 2)
    GROUP BY d.match_id, d.innings_number, d.match_date, d.batting_team, d.bowling_team, m.season
),
dis AS (
    SELECT DISTINCT d.match_id, d.innings_number
    FROM main_marts.fct_deliveries d
    JOIN main_marts.fct_match_results m USING (match_id)
    WHERE d.is_wicket AND d.wicket_player_out = '{PLAYER}'
      AND m.event_name ILIKE '%indian premier league%'
      AND d.innings_number IN (1, 2)
)
SELECT
    b.season, b.match_date, b.batting_team, b.bowling_team AS vs,
    b.runs,
    (d.match_id IS NULL) AS not_out,
    b.balls,
    round(b.runs * 100.0 / nullif(b.balls, 0), 1) AS sr,
    b.fours, b.sixes
FROM bat b
LEFT JOIN dis d USING (match_id, innings_number)
ORDER BY b.runs DESC
LIMIT 20
""")

,season,match_date,batting_team,vs,runs,not_out,balls,sr,fours,sixes
0,2014,2014-05-09,Royal Challengers Bangalore,Punjab Kings,29.0,False,23.0,126.1,4.0,0.0
1,2014,2014-04-26,Royal Challengers Bangalore,Rajasthan Royals,18.0,False,16.0,112.5,2.0,0.0
2,2014,2014-05-22,Royal Challengers Bangalore,Kolkata Knight Riders,12.0,True,8.0,150.0,1.0,0.0
3,2015,2015-04-22,Royal Challengers Bangalore,Chennai Super Kings,9.0,True,6.0,150.0,1.0,0.0
4,2014,2014-04-28,Royal Challengers Bangalore,Punjab Kings,8.0,False,17.0,47.1,0.0,0.0
5,2014,2014-05-20,Royal Challengers Bangalore,Sunrisers Hyderabad,6.0,False,6.0,100.0,1.0,0.0
6,2024,2024-03-23,Kolkata Knight Riders,Sunrisers Hyderabad,6.0,True,3.0,200.0,1.0,0.0
7,2014,2014-05-06,Royal Challengers Bangalore,Mumbai Indians,5.0,False,4.0,125.0,1.0,0.0
8,2014,2014-05-04,Royal Challengers Bangalore,Sunrisers Hyderabad,5.0,False,8.0,62.5,0.0,0.0
9,2024,2024-05-11,Kolkata Knight Riders,Mumbai Indians,2.0,True,2.0,100.0,0.0,0.0


## Bowling

In [ ]:
# Career IPL bowling summary
q(f"""
WITH bowl AS (
    SELECT
        d.match_id,
        d.innings_number,
        sum(CASE WHEN d.extras_wides = 0 AND d.extras_noballs = 0 THEN 1 ELSE 0 END)  AS legal_del,
        sum(d.runs_total - d.extras_byes - d.extras_legbyes)                           AS runs_conceded,
        sum(CASE WHEN d.is_wicket
                  AND d.wicket_kind NOT IN ('run out','retired hurt','retired out','obstructing the field')
             THEN 1 ELSE 0 END)                                                         AS wkts,
        sum(CASE WHEN d.extras_wides > 0 THEN 1 ELSE 0 END)                            AS wides,
        sum(CASE WHEN d.extras_noballs > 0 THEN 1 ELSE 0 END)                          AS no_balls
    FROM main_marts.fct_deliveries d
    JOIN main_marts.fct_match_results m USING (match_id)
    WHERE d.bowler = '{PLAYER}'
      AND m.event_name ILIKE '%indian premier league%'
      AND d.innings_number IN (1, 2)
    GROUP BY d.match_id, d.innings_number
),
maidens AS (
    SELECT d.match_id, d.innings_number, d.over_number,
        sum(d.runs_total - d.extras_byes - d.extras_legbyes) AS over_runs
    FROM main_marts.fct_deliveries d
    JOIN main_marts.fct_match_results m USING (match_id)
    WHERE d.bowler = '{PLAYER}'
      AND m.event_name ILIKE '%indian premier league%'
      AND d.innings_number IN (1, 2)
    GROUP BY d.match_id, d.innings_number, d.over_number
    HAVING sum(d.runs_total - d.extras_byes - d.extras_legbyes) = 0
       AND count(*) >= 6
)
SELECT
    count(DISTINCT b.match_id)                                                          AS mat,
    count(*)                                                                            AS inns,
    (floor(sum(b.legal_del) / 6)::integer || '.' || (sum(b.legal_del) % 6))::varchar   AS overs,
    (SELECT count(*) FROM maidens)                                                      AS mdns,
    sum(b.runs_conceded)                                                                AS runs,
    sum(b.wkts)                                                                         AS wkts,
    max_by(b.wkts || '/' || b.runs_conceded, b.wkts * 1000 - b.runs_conceded)          AS bbi,
    round(sum(b.runs_conceded)::double / nullif(sum(b.wkts), 0), 2)                    AS ave,
    round(sum(b.runs_conceded) * 6.0 / nullif(sum(b.legal_del), 0), 2)                 AS econ,
    round(sum(b.legal_del)::double / nullif(sum(b.wkts), 0), 2)                        AS sr,
    sum(CASE WHEN b.wkts >= 4 THEN 1 ELSE 0 END)                                       AS "4w",
    sum(CASE WHEN b.wkts >= 5 THEN 1 ELSE 0 END)                                       AS "5w"
FROM bowl b
""")

In [ ]:
# IPL bowling by season
q(f"""
WITH bowl AS (
    SELECT d.match_id, d.innings_number, m.season,
        sum(CASE WHEN d.extras_wides = 0 AND d.extras_noballs = 0 THEN 1 ELSE 0 END)  AS legal_del,
        sum(d.runs_total - d.extras_byes - d.extras_legbyes)                           AS runs_conceded,
        sum(CASE WHEN d.is_wicket
                  AND d.wicket_kind NOT IN ('run out','retired hurt','retired out','obstructing the field')
             THEN 1 ELSE 0 END)                                                         AS wkts
    FROM main_marts.fct_deliveries d
    JOIN main_marts.fct_match_results m USING (match_id)
    WHERE d.bowler = '{PLAYER}'
      AND m.event_name ILIKE '%indian premier league%'
      AND d.innings_number IN (1, 2)
    GROUP BY d.match_id, d.innings_number, m.season
)
SELECT
    season,
    count(DISTINCT match_id)                                                            AS mat,
    count(*)                                                                            AS inns,
    (floor(sum(legal_del) / 6)::integer || '.' || (sum(legal_del) % 6))::varchar       AS overs,
    sum(runs_conceded)                                                                  AS runs,
    sum(wkts)                                                                           AS wkts,
    max_by(wkts || '/' || runs_conceded, wkts * 1000 - runs_conceded)                  AS bbi,
    round(sum(runs_conceded)::double / nullif(sum(wkts), 0), 2)                         AS ave,
    round(sum(runs_conceded) * 6.0 / nullif(sum(legal_del), 0), 2)                     AS econ,
    round(sum(legal_del)::double / nullif(sum(wkts), 0), 2)                             AS sr,
    sum(CASE WHEN wkts >= 4 THEN 1 ELSE 0 END)                                          AS "4w",
    sum(CASE WHEN wkts >= 5 THEN 1 ELSE 0 END)                                          AS "5w"
FROM bowl
GROUP BY season
ORDER BY season
""")

In [ ]:
# Best bowling innings in IPL (top 20)
q(f"""
SELECT
    m.season,
    d.match_date,
    d.bowling_team,
    CASE WHEN m.team_1 = d.bowling_team THEN m.team_2 ELSE m.team_1 END AS vs,
    sum(CASE WHEN d.is_wicket
              AND d.wicket_kind NOT IN ('run out','retired hurt','retired out','obstructing the field')
         THEN 1 ELSE 0 END)                                              AS wkts,
    sum(d.runs_total - d.extras_byes - d.extras_legbyes)                 AS runs,
    (floor(sum(CASE WHEN d.extras_wides = 0 AND d.extras_noballs = 0
                    THEN 1 ELSE 0 END) / 6)::integer
     || '.' ||
     (sum(CASE WHEN d.extras_wides = 0 AND d.extras_noballs = 0
               THEN 1 ELSE 0 END) % 6))::varchar                         AS overs
FROM main_marts.fct_deliveries d
JOIN main_marts.fct_match_results m USING (match_id)
WHERE d.bowler = '{PLAYER}'
  AND m.event_name ILIKE '%indian premier league%'
  AND d.innings_number IN (1, 2)
GROUP BY m.season, d.match_id, d.innings_number, d.match_date, d.bowling_team, m.team_1, m.team_2
ORDER BY wkts DESC, runs
LIMIT 20
""")

In [ ]:
# Match-by-match bowling figures — 2025 IPL
q(f"""
SELECT
    d.match_date,
    CASE WHEN m.team_1 = d.bowling_team THEN m.team_2 ELSE m.team_1 END       AS vs,
    (floor(sum(CASE WHEN d.extras_wides = 0 AND d.extras_noballs = 0
                    THEN 1 ELSE 0 END) / 6)::integer
     || '.' ||
     (sum(CASE WHEN d.extras_wides = 0 AND d.extras_noballs = 0
               THEN 1 ELSE 0 END) % 6))::varchar                               AS overs,
    sum(d.runs_total - d.extras_byes - d.extras_legbyes)                       AS runs,
    sum(CASE WHEN d.is_wicket
              AND d.wicket_kind NOT IN ('run out','retired hurt','retired out','obstructing the field')
         THEN 1 ELSE 0 END)                                                    AS wkts,
    sum(CASE WHEN d.extras_wides  > 0 THEN 1 ELSE 0 END)                      AS wides,
    sum(CASE WHEN d.extras_noballs > 0 THEN 1 ELSE 0 END)                     AS nb,
    round(sum(d.runs_total - d.extras_byes - d.extras_legbyes) * 6.0
          / nullif(sum(CASE WHEN d.extras_wides = 0 AND d.extras_noballs = 0
                            THEN 1 ELSE 0 END), 0), 2)                         AS econ,
    d.match_id
FROM main_marts.fct_deliveries d
JOIN main_marts.fct_match_results m USING (match_id)
WHERE d.bowler = '{PLAYER}'
  AND m.event_name ILIKE '%indian premier league%'
  AND m.season = '2025'
  AND d.innings_number IN (1, 2)
GROUP BY d.match_id, d.match_date, d.innings_number, d.bowling_team, m.team_1, m.team_2
ORDER BY d.match_date
""")

In [ ]:
# Diagnostic: raw bowling innings for 2025 — shows all 11 rows and exposes any NULL bowling_team / match_date
q(f"""
SELECT
    d.match_id,
    d.innings_number,
    d.match_date,
    d.bowling_team,
    m.team_1,
    m.team_2,
    count(*)                                                                            AS deliveries,
    sum(CASE WHEN d.extras_wides = 0 AND d.extras_noballs = 0 THEN 1 ELSE 0 END)       AS legal_del,
    sum(d.runs_total - d.extras_byes - d.extras_legbyes)                               AS runs,
    sum(CASE WHEN d.is_wicket
              AND d.wicket_kind NOT IN ('run out','retired hurt','retired out','obstructing the field')
         THEN 1 ELSE 0 END)                                                             AS wkts
FROM main_marts.fct_deliveries d
JOIN main_marts.fct_match_results m USING (match_id)
WHERE d.bowler = '{PLAYER}'
  AND m.event_name ILIKE '%indian premier league%'
  AND m.season = '2025'
  AND d.innings_number IN (1, 2)
GROUP BY d.match_id, d.innings_number, d.match_date, d.bowling_team, m.team_1, m.team_2
ORDER BY d.match_date, d.innings_number
""")

In [33]:
# Career IPL fielding summary
q(f"""
SELECT
    sum(CASE WHEN wicket_kind = 'caught'
              AND wicket_fielders LIKE '%{PLAYER}%' THEN 1 ELSE 0 END)          AS catches,
    sum(CASE WHEN wicket_kind = 'caught and bowled'
              AND bowler = '{PLAYER}' THEN 1 ELSE 0 END)                         AS caught_and_bowled,
    sum(CASE WHEN wicket_kind = 'stumped'
              AND wicket_fielders LIKE '%{PLAYER}%' THEN 1 ELSE 0 END)          AS stumpings,
    sum(CASE WHEN wicket_kind = 'run out'
              AND wicket_fielders LIKE '%{PLAYER}%' THEN 1 ELSE 0 END)          AS run_outs
FROM main_marts.fct_deliveries d
JOIN main_marts.fct_match_results m USING (match_id)
WHERE m.event_name ILIKE '%indian premier league%'
  AND d.innings_number IN (1, 2)
  AND d.is_wicket
""")

,catches,caught_and_bowled,stumpings,run_outs
0,28.0,1.0,0.0,3.0


In [34]:
# IPL fielding by season
q(f"""
SELECT
    m.season,
    sum(CASE WHEN d.wicket_kind = 'caught'
              AND d.wicket_fielders LIKE '%{PLAYER}%' THEN 1 ELSE 0 END)        AS catches,
    sum(CASE WHEN d.wicket_kind = 'caught and bowled'
              AND d.bowler = '{PLAYER}' THEN 1 ELSE 0 END)                       AS caught_and_bowled,
    sum(CASE WHEN d.wicket_kind = 'stumped'
              AND d.wicket_fielders LIKE '%{PLAYER}%' THEN 1 ELSE 0 END)        AS stumpings,
    sum(CASE WHEN d.wicket_kind = 'run out'
              AND d.wicket_fielders LIKE '%{PLAYER}%' THEN 1 ELSE 0 END)        AS run_outs
FROM main_marts.fct_deliveries d
JOIN main_marts.fct_match_results m USING (match_id)
WHERE m.event_name ILIKE '%indian premier league%'
  AND d.innings_number IN (1, 2)
  AND d.is_wicket
GROUP BY m.season
ORDER BY m.season
""")

,season,catches,caught_and_bowled,stumpings,run_outs
0,2007/08,0.0,0.0,0.0,0.0
1,2009,0.0,0.0,0.0,0.0
2,2009/10,0.0,0.0,0.0,0.0
3,2011,0.0,0.0,0.0,0.0
4,2012,0.0,0.0,0.0,0.0
5,2013,0.0,0.0,0.0,0.0
6,2014,9.0,0.0,0.0,1.0
7,2015,7.0,0.0,0.0,0.0
8,2016,0.0,0.0,0.0,0.0
9,2017,0.0,0.0,0.0,0.0


In [36]:
# ── Diagnostic: all IPL innings for PLAYER ordered by date ───────────────────
q(f"""
WITH bat AS (
    SELECT d.match_id, d.innings_number, d.match_date, d.batting_team, m.season,
        sum(d.runs_batter) AS runs,
        sum(CASE WHEN NOT d.extras_wides THEN 1 ELSE 0 END) AS balls
    FROM main_marts.fct_deliveries d
    JOIN main_marts.fct_match_results m USING (match_id)
    WHERE d.batter = '{PLAYER}'
      AND m.event_name ILIKE '%indian premier league%'
      AND d.innings_number IN (1, 2)
    GROUP BY d.match_id, d.innings_number, d.match_date, d.batting_team, m.season
),
dis AS (
    SELECT DISTINCT d.match_id, d.innings_number, d.match_date, d.batting_team, m.season
    FROM main_marts.fct_deliveries d
    JOIN main_marts.fct_match_results m USING (match_id)
    WHERE d.is_wicket AND d.wicket_player_out = '{PLAYER}'
      AND m.event_name ILIKE '%indian premier league%'
      AND d.innings_number IN (1, 2)
),
all_inn AS (
    SELECT match_id, innings_number, match_date, batting_team, season FROM bat
    UNION
    SELECT match_id, innings_number, match_date, batting_team, season FROM dis
    UNION
    SELECT DISTINCT d.match_id, d.innings_number, d.match_date, d.batting_team, m.season
    FROM main_marts.fct_deliveries d
    JOIN main_marts.fct_match_results m USING (match_id)
    WHERE d.non_striker = '{PLAYER}'
      AND m.event_name ILIKE '%indian premier league%'
      AND d.innings_number IN (1, 2)
      AND NOT EXISTS (
          SELECT 1 FROM main_marts.fct_deliveries d2
          WHERE d2.match_id = d.match_id AND d2.batter = '{PLAYER}'
      )
)
SELECT
    a.season, a.match_date, a.batting_team,
    coalesce(b.runs,  0) AS runs,
    coalesce(b.balls, 0) AS balls,
    (d.match_id IS NOT NULL) AS dismissed
FROM all_inn a
LEFT JOIN bat b USING (match_id, innings_number)
LEFT JOIN dis d USING (match_id, innings_number)
ORDER BY a.match_date
""")

,season,match_date,batting_team,runs,balls,dismissed
0,2014,2014-04-24,Royal Challengers Bangalore,0.0,0.0,False
1,2014,2014-04-26,Royal Challengers Bangalore,18.0,16.0,True
2,2014,2014-04-28,Royal Challengers Bangalore,8.0,17.0,True
3,2014,2014-05-04,Royal Challengers Bangalore,5.0,8.0,True
4,2014,2014-05-06,Royal Challengers Bangalore,5.0,4.0,True
5,2014,2014-05-09,Royal Challengers Bangalore,29.0,23.0,True
6,2014,2014-05-20,Royal Challengers Bangalore,6.0,6.0,True
7,2014,2014-05-22,Royal Challengers Bangalore,12.0,8.0,False
8,2014,2014-05-24,Royal Challengers Bangalore,2.0,2.0,False
9,2015,2015-04-22,Royal Challengers Bangalore,9.0,6.0,False
